## Executive Summary

**Purpose:** Estimate power dissipation in motor cores

**What it does:** Calculate core (iron) losses in magnetic materials.

### Why It Matters
Thermal analysis needs core loss to calculate motor temperature rise and cooling requirements.

### Quick Start
**Inputs:** Flux density, frequency, material (steel grade or fitted parameters)

**Outputs:** Core loss in W/kg, decomposed into hysteresis and eddy current components

## How It Fits Into the Motor Design Workflow

**Upstream (depends on):** Used by thermal and efficiency analysis modules

**Downstream (used by):** See notebook connections

In [1]:
#| hide
import numpy as np
from scipy.optimize import curve_fit

## Theory: Core Loss Decomposition

Core loss arises from three mechanisms:
1. **Hysteresis loss** — Domain wall friction
2. **Eddy current loss** — Induced currents
3. **Excess loss** — Domain wall dynamics

### Steinmetz Model

$$P = k \cdot f^{\\alpha} \cdot \\hat{B}^{\\beta}$$

Simple two-exponent fit for sinusoidal excitation.

### Modified Steinmetz (iGSE)

$$P = k \cdot f^{\\alpha-1} \cdot (f \cdot \\hat{B})^{\\beta}$$

Improved for PWM drives and non-sinusoidal waveforms.

### Bertotti Three-Term

$$P_{total} = k_h f \\hat{B}^{\\beta} + k_e (f\\hat{B})^2 + k_a (f\\hat{B})^{1.5}$$

Physically-based separation into components.

In [2]:
#| export
def steinmetz(
    f: float | np.ndarray,
    B_peak: float | np.ndarray,
    k: float,
    alpha: float,
    beta: float,
) -> float | np.ndarray:
    r"""Classical Steinmetz core loss model.
    P = k * f^alpha * B^beta
    Args:
        f: Frequency (Hz)
        B_peak: Peak flux density (T)
        k: Material constant
        alpha: Frequency exponent (1.0–1.6)
        beta: Flux density exponent (1.6–2.2)
    Returns:
        Core loss (W/kg)
    """
    return k * (np.asarray(f) ** alpha) * (np.asarray(B_peak) ** beta)

In [3]:
#| export
def modified_steinmetz(
    f: float | np.ndarray,
    B_peak: float | np.ndarray,
    k: float,
    alpha: float,
    beta: float,
) -> float | np.ndarray:
    r"""Modified Steinmetz (iGSE) for PWM waveforms.
    P = k * f^(alpha-1) * (f*B)^beta
    Args:
        f: Frequency (Hz)
        B_peak: Peak flux density (T)
        k: Material constant
        alpha: Frequency exponent
        beta: Combined exponent
    Returns:
        Core loss (W/kg)
    """
    f = np.asarray(f)
    B = np.asarray(B_peak)
    return k * (f ** (alpha - 1)) * ((f * B) ** beta)

In [4]:
#| export
def bertotti(
    f: float | np.ndarray,
    B_peak: float | np.ndarray,
    k_h: float,
    k_e: float,
    k_a: float,
    alpha: float = 1.0,
    beta: float = 2.0,
) -> dict[str, float | np.ndarray]:
    r"""Bertotti three-term iron loss separation.
    Separates into hysteresis, eddy, and excess components.
    Args:
        f: Frequency (Hz)
        B_peak: Peak flux density (T)
        k_h: Hysteresis coefficient
        k_e: Eddy current coefficient
        k_a: Excess coefficient
        alpha: Frequency exponent (default 1.0)
        beta: Flux density exponent (default 2.0)
    Returns:
        Dict with 'hysteresis', 'eddy', 'excess', 'total' (W/kg)
    """
    f = np.asarray(f)
    B = np.asarray(B_peak)
    p_hyst = k_h * (f ** alpha) * (B ** beta)
    p_eddy = k_e * (f * B) ** 2
    p_exc  = k_a * (f * B) ** 1.5
    return {
        "hysteresis": p_hyst,
        "eddy": p_eddy,
        "excess": p_exc,
        "total": p_hyst + p_eddy + p_exc,
    }

In [5]:
#| export
def fit_steinmetz(
    f_arr: np.ndarray,
    B_arr: np.ndarray,
    loss_arr: np.ndarray,
) -> dict:
    """Fit Steinmetz parameters (k, α, β) to measured data."""
    def _func(X, k, alpha, beta):
        return steinmetz(X[0], X[1], k, alpha, beta)
    popt, _ = curve_fit(_func, (f_arr, B_arr), loss_arr,
                        p0=[0.01, 1.5, 2.5], maxfev=10000)
    fitted = _func((f_arr, B_arr), *popt)
    ss_res = np.sum((loss_arr - fitted) ** 2)
    r2 = 1.0 - ss_res / np.sum((loss_arr - np.mean(loss_arr)) ** 2)
    rmse = float(np.sqrt(np.mean((loss_arr - fitted) ** 2)))
    return {"k": popt[0], "alpha": popt[1], "beta": popt[2],
            "r2": float(r2), "rmse": rmse, "model": "Steinmetz"}

## Test

Verify iron loss calculations work correctly.

In [6]:
# Test Steinmetz
f = 400
B_peak = 1.0
k, alpha, beta = 0.05, 1.3, 1.8
P_steinmetz = steinmetz(f, B_peak, k, alpha, beta)
assert P_steinmetz > 0
print(f'✓ steinmetz() = {P_steinmetz:.2f} W/kg')

# Test Modified Steinmetz
P_mod = modified_steinmetz(f, B_peak, k, alpha, beta)
assert P_mod > 0
print(f'✓ modified_steinmetz() = {P_mod:.2f} W/kg')

# Test Bertotti
kh, ke, ka = 0.02, 0.0001, 0.001
loss_dict = bertotti(f, B_peak, kh, ke, ka)
assert loss_dict['total'] > 0
print(f'✓ bertotti() total = {loss_dict["total"]:.2f} W/kg')

print('\nAll tests passed!')

✓ steinmetz() = 120.68 W/kg
✓ modified_steinmetz() = 14564.51 W/kg
✓ bertotti() total = 32.00 W/kg

All tests passed!
